# CPRI Underwriting Test Notebook

In [ ]:

import json
from openai import OpenAI
from azure.storage.blob import BlobServiceClient

# ===== CONFIG =====
AZURE_STORAGE_CONNECTION_STRING = "YOUR_CONNECTION_STRING"
CONTAINER_NAME = "YOUR_CONTAINER_NAME"
OPENAI_API_KEY = "YOUR_OPENAI_KEY"
MODEL_NAME = "gpt-5-5"


In [ ]:

def read_blob_file(file_name):
    blob_service_client = BlobServiceClient.from_connection_string(
        AZURE_STORAGE_CONNECTION_STRING
    )
    container_client = blob_service_client.get_container_client(CONTAINER_NAME)
    blob_client = container_client.get_blob_client(file_name)

    data = blob_client.download_blob().readall()
    return json.loads(data)


In [ ]:

def build_payload(trade_risk, asset_desc, scoring):
    payload = {}

    traderisk_fields = [
        "deal_type","cover_type","policy_limit_usd","indemnity_percentage",
        "policy_tenor","brokerage_percentage","insured_party","obligor_name",
        "country_of_risk","soe_involvement","broker_firm","broker_contact",
        "commodity_type","bank_margin","payment_trigger"
    ]

    asset_fields = [
        "sponsor_name","prior_payment_history","repayment_structure",
        "offtake_agreement_quality","hedge_arrangements","security_collateral",
        "waiting_period_days"
    ]

    scoring_fields = [
        "country_tier","sovereign_rating","host_govt_track_record",
        "political_violence_level","bit_status","icsid_arbitration",
        "cend_risk","prior_claims_history","em_experience","erm_maturity",
        "corp_governance","abc_compliance","legal_enforceability",
        "eca_multilateral","financial_model_irr"
    ]

    for field in traderisk_fields:
        payload[field] = trade_risk.get(field)

    for field in asset_fields:
        payload[field] = asset_desc.get(field, payload.get(field))

    for field in scoring_fields:
        payload[field] = scoring.get(field)

    defaults = {
        "cover_type": "Non-Payment",
        "policy_tenor": "5 years",
        "indemnity_percentage": "90%",
        "waiting_period_days": 180
    }

    for k, v in defaults.items():
        if not payload.get(k):
            payload[k] = v

    return payload


In [ ]:

def build_prompt(data):
    return f"""
Analyze this CPRI submission:

— Submission & Deal —
• Deal Type: {data.get('deal_type')}
• Coverage Required: {data.get('cover_type')}
• Policy Limit (USD): {data.get('policy_limit_usd')}
• Indemnity %: {data.get('indemnity_percentage')}
• Waiting Period (days): {data.get('waiting_period_days')}
• Policy Tenor / Period: {data.get('policy_tenor')}
• Brokerage %: {data.get('brokerage_percentage')}

— Parties —
• Insured: {data.get('insured_party')}
• Obligor: {data.get('obligor_name')}
• Sponsor: {data.get('sponsor_name')}

— Country Risk —
• Country: {data.get('country_of_risk')}
• Rating: {data.get('sovereign_rating')}

— Transaction —
• Commodity: {data.get('commodity_type')}
• IRR: {data.get('financial_model_irr')}
"""


In [ ]:

client = OpenAI(api_key=OPENAI_API_KEY)

def run_analysis(prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a senior CPRI underwriter."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content


In [ ]:

# ===== RUN =====

traderisk = read_blob_file("traderisk.txt")
asset_desc = read_blob_file("assetdescription.txt")
scoring = read_blob_file("scoring_stub.json")

payload = build_payload(traderisk, asset_desc, scoring)
prompt = build_prompt(payload)

result = run_analysis(prompt)

print(result)
